# 1. Generating Ground Truth Data

**Goal of this stage:** turn *subjective* quality ("is search good?") into *measurable* quality.

To measure retrieval we need labelled pairs **(question -> correct document)**.
We have no real user logs yet, so we **generate them synthetically with the LLM**:

> For each FAQ document, ask the LLM to write 5 questions that this document
> would answer. The document we generated them from is, by construction, the
> correct answer for those questions.

So we get hundreds of test questions for free, each with a known-correct doc id.
This is the foundation for **every** metric in the module.

In [1]:
# Load the full FAQ (several courses), then keep only llm-zoomcamp docs.
from ingest import load_faq_data
documents = load_faq_data()

documents_llm = [d for d in documents if d["course"] == "llm-zoomcamp"]
documents = documents_llm
len(documents)

103

In [2]:
# Each document already has a stable `id` — that id becomes the LABEL.
doc = documents[0]
print(doc["id"]); print(doc["question"]); print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


## Structured output

We want a *Python object* back (a list of questions), not free text we then
parse. OpenAI offers `responses.parse(...)`; the **Anthropic** SDK achieves the
same by **forcing a tool call**: we register one tool whose `input_schema` is our
pydantic schema and force the model to call it. The tool arguments come back as a
validated dict. (See `evaluation_utils.llm_structured`.)

Define the output shape with pydantic:

In [3]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [4]:
# Connect to the LLM. anthropic.Anthropic() reads ANTHROPIC_AUTH_TOKEN +
# ANTHROPIC_BASE_URL (z.ai) from .env automatically.
import os, json
from dotenv import load_dotenv
import anthropic
load_dotenv()
client = anthropic.Anthropic()
print("model:", os.getenv("ANTHROPIC_MODEL"), "| base:", os.getenv("ANTHROPIC_BASE_URL"))

model: glm-5.2 | base: https://api.z.ai/api/anthropic


In [5]:
# Call the structured helper for ONE document -> get a Questions object + usage.
from evaluation_utils import llm_structured, calc_price

user_prompt = json.dumps(doc)
result, usage = llm_structured(client, data_gen_instructions, user_prompt, Questions)

print(result.questions)
calc_price(usage)

['Is it too late to enroll if I found the course late?', 'Am I able to sign up even though classes already started?', 'Can I hop in midway through or did I miss the deadline?', 'If I start now, will I still be eligible to get a certificate?', 'I came across this course after it began, is enrollment still open?']


{'input_cost': 0.000252, 'output_cost': 0.000369, 'total_cost': 0.000621}

In [6]:
# Turn the questions into labelled ground-truth records.
import pandas as pd
records = [{"question": q, "document": doc["id"]} for q in result.questions]
pd.DataFrame(records)

,question,document
0,Is it too late to enroll if I found the course...,74eb249bbf
1,Am I able to sign up even though classes alrea...,74eb249bbf
2,Can I hop in midway through or did I miss the ...,74eb249bbf
3,"If I start now, will I still be eligible to ge...",74eb249bbf
4,"I came across this course after it began, is e...",74eb249bbf


## Generate for all documents (batch + parallel)

`generate_ground_truth` wraps one doc -> 5 labelled records. We run it across
documents in parallel with `map_progress` (a ThreadPoolExecutor + tqdm bar) and
track cost.

> **Why regenerate instead of the shipped `ground_truth-new.csv`?** The live FAQ
> has drifted since that file was made (docs removed, new ones added) — ~20% of
> its doc ids no longer exist, which would cap our hit rate. **Evaluation data
> goes stale.** Regenerating against the *current* FAQ keeps the whole pipeline
> self-consistent, so we generate for all live docs here.

In [7]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import llm_structured_retry, map_progress, calc_total_price

N_GEN = len(documents)  # generate for ALL live docs so ids match the current FAQ

def generate_ground_truth(doc):
    out, usage = llm_structured_retry(
        client, data_gen_instructions, json.dumps(doc), Questions
    )
    records = [{"question": q, "document": doc["id"]} for q in out.questions]
    return records, usage

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents[:N_GEN], generate_ground_truth)

ground_truth, usages = [], []
for recs, u in results:
    ground_truth.extend(recs)
    usages.append(u)

print("generated questions:", len(ground_truth))
print("cost: $%.6f" % calc_total_price(usages))

  0%|          | 0/103 [00:00<?, ?it/s]

generated questions: 515
cost: $0.075880


In [8]:
# Save the freshly generated ground truth. The next notebooks read THIS file.
pd.DataFrame(ground_truth).to_csv("data/ground_truth.csv", index=False)
print("saved", len(ground_truth), "rows to data/ground_truth.csv")

saved 515 rows to data/ground_truth.csv


**Takeaway:** we now have labelled `(question, document)` pairs. The next
notebook asks the search engine each question and checks whether it returns the
labelled document.